In [ ]:
import dgl.nn as dglnn
from dgl import from_networkx
import torch.nn as nn
import torch as th
import torch.nn.functional as F
import dgl.function as fn
from dgl.data.utils import load_graphs
import networkx as nx
import pandas as pd
import socket
import struct
import random
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
data = pd.read_csv('bot_iot.csv')

In [ ]:
data.drop(columns=['subcategory','pkSeqID','stime','flgs','category','state','proto','seq'],inplace=True)

In [ ]:
data.rename(columns={"attack": "label"},inplace = True)

In [ ]:
data.label.value_counts()

In [ ]:
attack = data[data['label'] == 1].sample(frac=0.1)

In [ ]:
normal = data[data['label'] == 0]

In [ ]:
data = pd.concat([normal,attack])

In [ ]:
data.label.value_counts()

In [ ]:
#le = LabelEncoder()
#le.fit_transform(data.label.values)
#data['label'] = le.transform(data['label'])

In [ ]:
data['saddr'] = data.saddr.apply(str)
data['sport'] = data.sport.apply(str)
data['daddr'] = data.daddr.apply(str)
data['dport'] = data.dport.apply(str)

In [ ]:
data['saddr'] = data.saddr.apply(lambda x: socket.inet_ntoa(struct.pack('>I', random.randint(0xac100001, 0xac1f0001))))

In [ ]:
data['saddr'] = data['saddr'] + ':' + data['sport']
data['daddr'] = data['daddr'] + ':' + data['dport']

In [ ]:
data.drop(columns=['sport','dport'],inplace=True)
label_ground_truth = data[["saddr", "daddr","label"]]
data = pd.get_dummies(data, columns = ['flgs_number','state_number', 'proto_number'])

In [ ]:
data = data.reset_index()
data.replace([np.inf, -np.inf], np.nan,inplace = True)

In [ ]:
data.fillna(0,inplace = True)

In [ ]:
label_ground_truth = data[["saddr", "daddr","label"]]

In [ ]:
data.drop(columns=['index'],inplace=True)

In [ ]:
data

In [ ]:
scaler = StandardScaler()
cols_to_norm = list(set(list(data.iloc[:, 2:].columns ))  - set(list(['label'])) )
data[cols_to_norm] = scaler.fit_transform(data[cols_to_norm])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
     data, label_ground_truth, test_size=0.3, random_state=42, stratify=label_ground_truth.label)

In [ ]:
X_train['h'] = X_train[ cols_to_norm ].values.tolist()

In [ ]:
#from dgl.data.utils import load_graphs   
#G = load_graphs("./data.bin")[0][0]

In [ ]:
G = nx.from_pandas_edgelist(X_train, "saddr", "daddr", ['h','label'], create_using= nx.MultiGraph())

In [ ]:
G = G.to_directed()

In [ ]:
G = from_networkx(G,edge_attrs=['h','label'])

In [ ]:
#from dgl.data.utils import save_graphs
#save_graphs("./data.bin", [G])


In [ ]:
G.ndata['h'] = th.ones(G.num_nodes(), G.edata['h'].shape[1]) 

In [ ]:
G.edata['train_mask'] = th.ones(len(G.edata['h']), dtype= th.bool)

In [ ]:
#G = load_graphs("./bot_train_G.bin") [0][0]

In [ ]:
G.ndata['h'] = th.ones(G.num_nodes(), G.edata['h'].shape[1])

In [ ]:
G.edata['train_mask'] = th.ones(len(G.edata['h']), dtype=th.bool)

In [ ]:
G.ndata['h'] = th.reshape(G.ndata['h'], (G.ndata['h'].shape[0], 1, G.ndata['h'].shape[1]))
G.edata['h'] = th.reshape(G.edata['h'], (G.edata['h'].shape[0], 1, G.edata['h'].shape[1]))

In [ ]:
class MLPPredictor(nn.Module):
    def __init__(self, in_features, out_classes):
        super().__init__()
        self.W = nn.Linear(in_features * 2, out_classes)

    def apply_edges(self, edges):
        h_u = edges.src['h']
        h_v = edges.dst['h']
        global score
        global emb
        emb = th.cat([h_u, h_v], 1)
        score = self.W(th.cat([h_u, h_v], 1))
        return {'score': score}

    def forward(self, graph, h):
        with graph.local_scope():
            graph.ndata['h'] = h
            graph.apply_edges(self.apply_edges)
            return graph.edata['score']

In [ ]:
G.ndata['h'].shape

In [ ]:
def compute_accuracy(pred, labels):
    return (pred.argmax(1) == labels).float().mean().item()

In [ ]:
class SAGELayer(nn.Module):
    def __init__(self, ndim_in, edims, ndim_out, activation):
        super(SAGELayer, self).__init__()
        ### force to outut fix dimensions
        self.W_msg = nn.Linear(ndim_in + edims, ndim_out)
        ### apply weight
        self.W_apply = nn.Linear(ndim_in + ndim_out, ndim_out)
        self.activation = activation

    def message_func(self, edges):
        return {'m': self.W_msg(th.cat([edges.src['h'], edges.data['h']], 2))}

    def forward(self, g_dgl, nfeats, efeats):
        with g_dgl.local_scope():
            g = g_dgl
            g.ndata['h'] = nfeats
            g.edata['h'] = efeats
            # Eq4
            g.update_all(self.message_func, fn.mean('m', 'h_neigh'))
            # Eq5          
            g.ndata['h'] = F.relu(self.W_apply(th.cat([g.ndata['h'], g.ndata['h_neigh']], 2)))
            return g.ndata['h']


class SAGE(nn.Module):
    def __init__(self, ndim_in, ndim_out, edim, activation, dropout):
        super(SAGE, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(SAGELayer(ndim_in, edim, 128, activation))
        self.layers.append(SAGELayer(128, edim, ndim_out, activation))
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, g, nfeats, efeats):
        for i, layer in enumerate(self.layers):
            if i != 0:
                nfeats = self.dropout(nfeats)
            nfeats = layer(g, nfeats, efeats)
        return nfeats.sum(1)

In [ ]:
class Model(nn.Module):
    def __init__(self, ndim_in, ndim_out, edim, activation, dropout):
        super().__init__()
        self.gnn = SAGE(ndim_in, ndim_out, edim, activation, dropout)
        self.pred = MLPPredictor(ndim_out, 2)
    def forward(self, g, nfeats, efeats):
        h = self.gnn(g, nfeats, efeats)
        return self.pred(g, h)

In [ ]:
from sklearn.utils import class_weight
class_weights = class_weight.compute_class_weight('balanced',
                                                 classes=np.unique(G.edata['label'].cpu().numpy()),
                                                 y=G.edata['label'].cpu().numpy())

In [ ]:
class_weights = th.FloatTensor(class_weights)#.cuda()
criterion = nn.CrossEntropyLoss(weight = class_weights)

In [ ]:
#G = G.to('cuda:0')

In [ ]:
G.device

In [ ]:
G.ndata['h'].device  

In [ ]:
G.edata['h'].device  

In [ ]:
node_features = G.ndata['h']
edge_features = G.edata['h']

edge_label = G.edata['label']
train_mask = G.edata['train_mask']

model = Model(G.ndata['h'].shape[2], 128, G.ndata['h'].shape[2], F.relu, 0.2)#.cuda()

opt = th.optim.Adam(model.parameters())

for epoch in range(1,50):
    pred = model(G, node_features,edge_features)#.cuda()
    loss = criterion(pred[train_mask] ,edge_label[train_mask])
    opt.zero_grad()
    loss.backward()
    opt.step()
    if epoch % 50 == 0:
        print('Epoch:', epoch ,' Training acc:', compute_accuracy(pred[train_mask], edge_label[train_mask]))

In [ ]:
#model.load_state_dict(th.load('bot_binary.pt'))
#model.eval()

In [ ]:
X_test['h'] = X_test[ cols_to_norm ].values.tolist()

In [ ]:
#G_test = load_graphs("bot_test_G.bin") [0][0]
G_test = nx.from_pandas_edgelist(X_test, "saddr", "daddr", ['h','label'],create_using=nx.MultiGraph())
G_test = G_test.to_directed()
G_test = from_networkx(G_test,edge_attrs=['h','label'] )
actual = G_test.edata.pop('label')
G_test.ndata['feature'] = th.ones(G_test.num_nodes(), 53)

In [ ]:
G_test.ndata['feature'] = th.reshape(G_test.ndata['feature'], (G_test.ndata['feature'].shape[0], 1, G_test.ndata['feature'].shape[1]))

In [ ]:
G_test.edata['h'] = th.reshape(G_test.edata['h'], (G_test.edata['h'].shape[0], 1, G_test.edata['h'].shape[1]))

In [ ]:
G_test = G_test#.to('cuda:0')

In [ ]:
#th.cuda.empty_cache()

In [ ]:
import timeit
start_time = timeit.default_timer()
node_features_test = G_test.ndata['feature']
edge_features_test = G_test.edata['h']
test_pred = model(G_test, node_features_test, edge_features_test)#.cuda()
elapsed = timeit.default_timer() - start_time

In [ ]:
print(str(elapsed) + ' seconds')

In [ ]:
test_pred = test_pred.argmax(1)

In [ ]:
test_pred = th.Tensor.cpu(test_pred).detach().numpy()

In [ ]:
actual = ["Normal" if i == 0 else "Attack" for i in actual]
test_pred = ["Normal" if i == 0 else "Attack" for i in test_pred]

In [ ]:
def plot_confusion_matrix(cm,
                          target_names,
                          title='Confusion matrix',
                          cmap=None,
                          normalize=True):
    
    import matplotlib.pyplot as plt
    import numpy as np
    import itertools

    accuracy = np.trace(cm) / float(np.sum(cm))
    misclass = 1 - accuracy

    if cmap is None:
        cmap = plt.get_cmap('Blues')

    plt.figure(figsize=(12, 12))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()

    if target_names is not None:
        tick_marks = np.arange(len(target_names))
        plt.xticks(tick_marks, target_names, rotation=45)
        plt.yticks(tick_marks, target_names)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]


    thresh = cm.max() / 1.5 if normalize else cm.max() / 2
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        if normalize:
            plt.text(j, i, "{:0.4f}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")
        else:
            plt.text(j, i, "{:,}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")


    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label\naccuracy={:0.4f}; misclass={:0.4f}'.format(accuracy, misclass))
    plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix

plot_confusion_matrix(cm = confusion_matrix(actual, test_pred), 
                      normalize    = False,
                      target_names = np.unique(actual),
                      title        = "Confusion Matrix")

In [ ]:
from sklearn.metrics import classification_report
target_names = np.unique(actual)
print(classification_report(actual, test_pred, target_names=target_names, digits=4))

In [ ]:
emb

In [ ]:
emb_num = emb.cpu().detach().numpy()


In [ ]:
np.save('emb_bin.npy',emb_num)

In [ ]:
emb_viz = np.load('emb_viz.npy')

In [ ]:
emb_viz

In [ ]:
df_umap = pd.DataFrame(emb_viz, columns=['comp1', 'comp2'])


In [ ]:
df_umap['label'] = actual


In [ ]:
df_umap

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt 

plt.figure(figsize=(8,8));
sns.scatterplot(x='comp1', y='comp2', data=df_umap, hue='label');
plt.legend(loc='upper left', frameon=False)

In [ ]:
np.save('raw.npy',X_test.drop(columns=['saddr','daddr','h']).to_numpy())

In [ ]:
raw_viz = np.load('raw_viz.npy')

In [ ]:
raw_viz

In [ ]:
df_umap = pd.DataFrame(raw_viz, columns=['comp1', 'comp2'])

In [ ]:
y_label =  ["Normal" if i == 0 else "Attack" for i in y_test.label]

In [ ]:
df_umap['label'] = y_label

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt 

plt.figure(figsize=(8,8));
sns.scatterplot(x='comp1', y='comp2', data=df_umap, hue='label');
plt.legend(loc='upper left', frameon=False)